## Imports and dataset discovery


In [1]:
# ============================================
# Cell 1 — Imports and dataset discovery
# ============================================
import os, time, math, random, json
from pathlib import Path
from collections import defaultdict

import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'CUDA: {torch.cuda.get_device_name(0)}')

INPUT_ROOT = Path('/kaggle/input')
print(f'\nAttached datasets:')
for d in INPUT_ROOT.iterdir():
    print(f'  {d.name}')
    for f in d.rglob('*'):
        if f.is_file():
            print(f'    {f.stat().st_size/1024**2:7.1f} MB  {f.name}')

Device: cuda
CUDA: Tesla T4

Attached datasets:
  datasets
      241.7 MB  training_objectdataset_augmentedrot_scale75.h5
       60.1 MB  test_objectdataset_augmentedrot_scale75.h5
        6.8 MB  pointnetpp_msg_domain_aug_best.pt
      115.4 MB  modelnet40_train.npz
       26.2 MB  s3dis_finetune_cache.npz


## Load checkpoint and all 3 training datasets



In [2]:
# ============================================
# Cell 2 — Load checkpoint and all 3 training datasets
# ============================================
def find_file(filename_part, root=INPUT_ROOT):
    matches = [f for f in root.rglob('*') if filename_part in f.name and f.is_file()]
    return matches[0] if matches else None

ckpt_path     = find_file('pointnetpp_msg_domain_aug_best')
s3dis_path    = find_file('s3dis_finetune_cache')
mn40_path     = find_file('modelnet40_train')
so_train_path = find_file('training_objectdataset_augmentedrot_scale75')
so_test_path  = find_file('test_objectdataset_augmentedrot_scale75')

for label, p in [('Checkpoint', ckpt_path),
                 ('ModelNet40 train', mn40_path),
                 ('S3DIS cache', s3dis_path),
                 ('ScanObjectNN train', so_train_path),
                 ('ScanObjectNN test', so_test_path)]:
    print(f'{label:25s}: {p}')
    assert p is not None, f'{label} not found!'

# Load ModelNet40 train (Y-up native, MN40 labels)
mn40 = np.load(mn40_path, allow_pickle=True)
mn40_keys = list(mn40.keys())
mn40_data   = mn40[mn40_keys[0]].astype(np.float32)
mn40_labels = mn40[mn40_keys[1]].astype(np.int64).flatten()
print(f'\nMN40 train: data {mn40_data.shape}, labels classes 0-{mn40_labels.max()}')

# Load S3DIS (Y-up, MN40 indices)
s3dis = np.load(s3dis_path, allow_pickle=True)
s3dis_data   = s3dis['points'].astype(np.float32)
s3dis_labels = s3dis['labels'].astype(np.int64)
print(f'S3DIS:      data {s3dis_data.shape}, labels classes {sorted(set(s3dis_labels.tolist()))}')

# Load ScanObjectNN train + test (Z-up native, 15-class)
def load_h5_so(p):
    with h5py.File(p, 'r') as f:
        return f['data'][:].astype(np.float32), f['label'][:].astype(np.int64).flatten()

so_train_data, so_train_labels = load_h5_so(so_train_path)
so_test_data,  so_test_labels  = load_h5_so(so_test_path)
print(f'SO train:   data {so_train_data.shape}, labels classes 0-{so_train_labels.max()}')
print(f'SO test:    data {so_test_data.shape},  labels classes 0-{so_test_labels.max()}')

Checkpoint               : /kaggle/input/datasets/vathsal05/multi-domain-finetune-inputs/pointnetpp_msg_domain_aug_best.pt
ModelNet40 train         : /kaggle/input/datasets/vathsal05/multi-domain-finetune-inputs/modelnet40_train.npz
S3DIS cache              : /kaggle/input/datasets/vathsal05/multi-domain-finetune-inputs/s3dis_finetune_cache.npz
ScanObjectNN train       : /kaggle/input/datasets/vathsal05/scanobjectnn-pb-t50-rs/training_objectdataset_augmentedrot_scale75.h5
ScanObjectNN test        : /kaggle/input/datasets/vathsal05/scanobjectnn-pb-t50-rs/test_objectdataset_augmentedrot_scale75.h5

MN40 train: data (9843, 1024, 3), labels classes 0-39
S3DIS:      data (2232, 1024, 3), labels classes [4, 8, 13, 30, 33]
SO train:   data (11416, 2048, 3), labels classes 0-14
SO test:    data (2882, 2048, 3),  labels classes 0-14


##  Map ScanObjectNN labels into MN40 space + axis swap


In [3]:
# ============================================
# Cell 3 — Convert ScanObjectNN to MN40 label space + Y-up
# ============================================
MODELNET40_CLASSES = [
    'airplane', 'bathtub', 'bed', 'bench', 'bookshelf', 'bottle', 'bowl', 'car',
    'chair', 'cone', 'cup', 'curtain', 'desk', 'door', 'dresser', 'flower_pot',
    'glass_box', 'guitar', 'keyboard', 'lamp', 'laptop', 'mantel', 'monitor',
    'night_stand', 'person', 'piano', 'plant', 'radio', 'range_hood', 'sink',
    'sofa', 'stairs', 'stool', 'table', 'tent', 'toilet', 'tv_stand', 'vase',
    'wardrobe', 'xbox'
]
MN40_NAME_TO_IDX = {n: i for i, n in enumerate(MODELNET40_CLASSES)}

# ScanObjectNN idx -> MN40 idx (None means EXCLUDE)
SO_TO_MN40 = {
    0: None,                                    # bag
    1: None,                                    # bin
    2: None,                                    # box
    3:  MN40_NAME_TO_IDX['wardrobe'],           # cabinet
    4:  MN40_NAME_TO_IDX['chair'],
    5:  MN40_NAME_TO_IDX['desk'],
    6:  MN40_NAME_TO_IDX['monitor'],            # display
    7:  MN40_NAME_TO_IDX['door'],
    8:  MN40_NAME_TO_IDX['bookshelf'],          # shelf
    9:  MN40_NAME_TO_IDX['table'],
    10: MN40_NAME_TO_IDX['bed'],
    11: None,                                   # pillow
    12: MN40_NAME_TO_IDX['sink'],
    13: MN40_NAME_TO_IDX['sofa'],
    14: MN40_NAME_TO_IDX['toilet'],
}

def filter_and_remap_so(data, labels):
    mn40 = np.array([SO_TO_MN40.get(int(l), None) for l in labels])
    keep = np.array([x is not None for x in mn40])
    return data[keep], mn40[keep].astype(np.int64)

# Filter to mappable
so_train_data_kept, so_train_labels_mn40 = filter_and_remap_so(so_train_data, so_train_labels)
so_test_data_kept,  so_test_labels_mn40  = filter_and_remap_so(so_test_data,  so_test_labels)

# Apply Y<->Z axis swap (ScanObjectNN is Z-up, target Y-up)
so_train_data_kept = so_train_data_kept[:, :, [0, 2, 1]]
so_test_data_kept  = so_test_data_kept[:, :, [0, 2, 1]]

print(f'ScanObjectNN train mappable + axis-swapped: {so_train_data_kept.shape}')
print(f'ScanObjectNN test  mappable + axis-swapped: {so_test_data_kept.shape}')

# Combined class distribution
combined_labels = np.concatenate([mn40_labels, so_train_labels_mn40, s3dis_labels])
print(f'\nCombined training data: {len(combined_labels)} samples')
unique, counts = np.unique(combined_labels, return_counts=True)
print(f'Distribution across MN40 classes:')
for u, c in sorted(zip(unique, counts), key=lambda x: -x[1])[:15]:
    print(f'  {u:2d} {MODELNET40_CLASSES[u]:12s}: {c:5d}')
print(f'  ... (showing top 15 of {len(unique)} classes)')

ScanObjectNN train mappable + axis-swapped: (9513, 2048, 3)
ScanObjectNN test  mappable + axis-swapped: (2362, 2048, 3)

Combined training data: 21588 samples
Distribution across MN40 classes:
   8 chair       :  3579
   4 bookshelf   :  2022
  30 sofa        :  1782
  33 table       :  1615
  38 wardrobe    :  1431
  13 door        :  1417
  22 monitor     :  1143
   2 bed         :  1079
  12 desk        :   792
  35 toilet      :   669
   0 airplane    :   626
  29 sink        :   597
  37 vase        :   475
   5 bottle      :   335
  21 mantel      :   284
  ... (showing top 15 of 40 classes)


## Augmentation utilities

In [4]:
# ============================================
# Cell 4 — Augmentation utilities (Y-up target)
# ============================================
def random_rotation_y(points):
    '''Random rotation around the Y (vertical) axis — Y-up convention.'''
    theta = np.random.uniform(0, 2 * np.pi)
    cos, sin = np.cos(theta), np.sin(theta)
    R = np.array([[ cos, 0, sin],
                  [ 0,   1, 0  ],
                  [-sin, 0, cos]], dtype=np.float32)
    return points @ R.T

def random_scale(points, lo=0.8, hi=1.25):
    return points * np.random.uniform(lo, hi)

def random_translate(points, t=0.1):
    return points + np.random.uniform(-t, t, size=(1, 3)).astype(np.float32)

def jitter(points, sigma=0.01, clip=0.05):
    noise = np.clip(sigma * np.random.randn(*points.shape), -clip, clip).astype(np.float32)
    return points + noise

def random_point_dropout(points, max_dropout=0.125):
    dropout_ratio = np.random.uniform(0, max_dropout)
    drop_idx = np.where(np.random.random(len(points)) <= dropout_ratio)[0]
    if len(drop_idx) > 0:
        points = points.copy()
        points[drop_idx] = points[0]
    return points

def normalize_unit_sphere(points):
    points = points - points.mean(axis=0, keepdims=True)
    max_dist = np.linalg.norm(points, axis=1).max()
    return points / (max_dist + 1e-8)

print('Augmentation utilities defined.')

Augmentation utilities defined.


## Combined dataset class

In [5]:
# ============================================
# Cell 5 — Combined dataset (3 sources, Y-up, MN40 labels)
# ============================================
class PointDataset(Dataset):
    '''Generic point cloud dataset with optional downsampling and augmentation.'''
    def __init__(self, data, labels, n_points=1024, training=True, name=''):
        self.data = data
        self.labels = labels
        self.n_points = n_points
        self.training = training
        self.name = name

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        points = self.data[idx].copy()                          # (N, 3)
        # Random subsample if needed (fast, matches training)
        if points.shape[0] != self.n_points:
            choice = np.random.choice(points.shape[0], self.n_points, replace=False)
            points = points[choice]
        points = normalize_unit_sphere(points)
        if self.training:
            points = random_rotation_y(points)
            points = random_scale(points)
            points = random_translate(points)
            points = jitter(points)
            points = random_point_dropout(points)
            points = normalize_unit_sphere(points)
        return torch.from_numpy(points.astype(np.float32)), int(self.labels[idx])

N_POINTS   = 1024
BATCH_SIZE = 32

# Build per-source datasets
mn40_train_ds   = PointDataset(mn40_data,           mn40_labels,           n_points=N_POINTS, training=True,  name='MN40')
s3dis_train_ds  = PointDataset(s3dis_data,          s3dis_labels,          n_points=N_POINTS, training=True,  name='S3DIS')
so_train_ds_obj = PointDataset(so_train_data_kept,  so_train_labels_mn40,  n_points=N_POINTS, training=True,  name='SO_train')

# Test sets (evaluation only — no augmentation)
so_test_ds      = PointDataset(so_test_data_kept,   so_test_labels_mn40,   n_points=N_POINTS, training=False, name='SO_test')

# Combine training sources into one big dataset
combined_train_ds = ConcatDataset([mn40_train_ds, so_train_ds_obj, s3dis_train_ds])

train_loader = DataLoader(combined_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, drop_last=True, pin_memory=True)
test_loader  = DataLoader(so_test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, drop_last=False, pin_memory=True)

print(f'Combined train: {len(combined_train_ds)} samples ({len(train_loader)} batches/epoch)')
print(f'  - MN40:         {len(mn40_train_ds)}')
print(f'  - ScanObjectNN: {len(so_train_ds_obj)}')
print(f'  - S3DIS:        {len(s3dis_train_ds)}')
print(f'ScanObjectNN test (eval-only): {len(so_test_ds)}')

Combined train: 21588 samples (674 batches/epoch)
  - MN40:         9843
  - ScanObjectNN: 9513
  - S3DIS:        2232
ScanObjectNN test (eval-only): 2362


## PointNet++ MSG architecture

In [6]:
# ============================================
# Cell 6 — PointNet++ MSG (40-class)
# ============================================
def farthest_point_sample(xyz, npoint):
    B, N, _ = xyz.shape
    dev = xyz.device
    centroids = torch.zeros(B, npoint, dtype=torch.long, device=dev)
    distance = torch.full((B, N), float("inf"), device=dev)
    farthest = torch.randint(0, N, (B,), dtype=torch.long, device=dev)
    batch_idx = torch.arange(B, dtype=torch.long, device=dev)
    for i in range(npoint):
        centroids[:, i] = farthest
        cxyz = xyz[batch_idx, farthest, :].unsqueeze(1)
        dist = ((xyz - cxyz) ** 2).sum(dim=-1)
        distance = torch.minimum(distance, dist)
        farthest = distance.argmax(dim=-1)
    return centroids


def index_points(points, idx):
    B = points.shape[0]
    vs = list(idx.shape); vs[1:] = [1]*(len(vs)-1)
    rs = list(idx.shape); rs[0] = 1
    bi = torch.arange(B, dtype=torch.long, device=points.device).view(vs).repeat(rs)
    return points[bi, idx, :]


def ball_query(radius, nsample, xyz, new_xyz):
    B, N, _ = xyz.shape
    _, S, _ = new_xyz.shape
    dev = xyz.device
    gi = torch.arange(N, dtype=torch.long, device=dev).view(1, 1, N).repeat(B, S, 1)
    sd = ((new_xyz.unsqueeze(2) - xyz.unsqueeze(1)) ** 2).sum(dim=-1)
    gi[sd > radius ** 2] = N
    gi = gi.sort(dim=-1)[0][:, :, :nsample]
    gf = gi[:, :, 0:1].repeat(1, 1, nsample); gi[gi == N] = gf[gi == N]
    return gi


class SetAbstractionMSG(nn.Module):
    def __init__(self, npoint, radii, nsamples, in_channel, mlps):
        super().__init__()
        self.npoint, self.radii, self.nsamples = npoint, radii, nsamples
        self.conv_blocks, self.bn_blocks = nn.ModuleList(), nn.ModuleList()
        for mlp in mlps:
            convs, bns = nn.ModuleList(), nn.ModuleList()
            last = in_channel + 3
            for c in mlp:
                convs.append(nn.Conv2d(last, c, 1)); bns.append(nn.BatchNorm2d(c)); last = c
            self.conv_blocks.append(convs); self.bn_blocks.append(bns)

    def forward(self, xyz, features=None):
        fps = farthest_point_sample(xyz, self.npoint)
        new_xyz = index_points(xyz, fps); outs = []
        for i, (r, k) in enumerate(zip(self.radii, self.nsamples)):
            nn_idx = ball_query(r, k, xyz, new_xyz)
            g_xyz = index_points(xyz, nn_idx) - new_xyz.unsqueeze(2)
            if features is not None:
                g = torch.cat([g_xyz, index_points(features, nn_idx)], dim=-1)
            else:
                g = g_xyz
            g = g.permute(0, 3, 1, 2).contiguous()
            for conv, bn in zip(self.conv_blocks[i], self.bn_blocks[i]):
                g = F.relu(bn(conv(g)))
            outs.append(g.max(dim=-1)[0])
        return new_xyz, torch.cat(outs, dim=1).permute(0, 2, 1).contiguous()


class GlobalSetAbstraction(nn.Module):
    def __init__(self, in_channel, mlp):
        super().__init__()
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        last = in_channel
        for c in mlp:
            self.convs.append(nn.Conv1d(last, c, 1)); self.bns.append(nn.BatchNorm1d(c)); last = c

    def forward(self, xyz, features):
        x = torch.cat([xyz, features], dim=-1).permute(0, 2, 1)
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x)))
        return x.max(dim=-1)[0]


class PointNetPlusPlusMSG(nn.Module):
    def __init__(self, num_classes=40, dropout=0.5):
        super().__init__()
        self.sa1 = SetAbstractionMSG(512, [0.1, 0.2, 0.4], [16, 32, 128], 0,
                                     [[32, 32, 64], [64, 64, 128], [64, 96, 128]])
        self.sa2 = SetAbstractionMSG(128, [0.2, 0.4, 0.8], [32, 64, 128], 320,
                                     [[64, 64, 128], [128, 128, 256], [128, 128, 256]])
        self.sa_global = GlobalSetAbstraction(640 + 3, [256, 512, 1024])
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512,  256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,  num_classes),
        )

    def forward(self, xyz):
        l1_xyz, l1_f = self.sa1(xyz, None)
        l2_xyz, l2_f = self.sa2(l1_xyz, l1_f)
        return self.classifier(self.sa_global(l2_xyz, l2_f))


print('PointNet++ MSG defined (40-class).')

PointNet++ MSG defined (40-class).


## Load pretrained checkpoint + EMA + train/eval helpers

In [7]:
# ============================================
# Cell 7 — Load domain_aug checkpoint + helpers
# ============================================
def load_pretrained(ckpt_path, model, device):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = None
    for key in ('ema_state_dict', 'ema', 'model_state_dict', 'state_dict', 'model'):
        if isinstance(ckpt, dict) and key in ckpt and isinstance(ckpt[key], dict):
            state = ckpt[key]
            print(f'  Loading [{key}] from {ckpt_path.name}')
            break
    if state is None:
        state = ckpt
        print(f'  Loading raw state-dict from {ckpt_path.name}')
    state = {k.replace('module.', '', 1) if k.startswith('module.') else k: v
             for k, v in state.items()}
    missing, unexpected = model.load_state_dict(state, strict=False)
    n_total = len(list(model.state_dict().keys()))
    n_loaded = n_total - len(missing)
    print(f'  Loaded {n_loaded}/{n_total} weight tensors')
    if n_loaded < n_total * 0.9:
        raise RuntimeError(f'Only {n_loaded}/{n_total} weights loaded — checkpoint mismatch!')


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)
            else:
                self.shadow[k].copy_(v.detach())

    def apply(self, model):
        model.load_state_dict(self.shadow, strict=True)


def train_one_epoch(model, loader, optimizer, scaler, loss_fn, ema, device):
    model.train()
    total_loss, total, correct = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            logits = model(xb)
            loss   = loss_fn(logits, yb)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        ema.update(model)
        total_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=-1)
        correct += (preds == yb).sum().item()
        total += xb.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    per_class = defaultdict(lambda: [0, 0])
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        logits = model(xb)
        preds = logits.argmax(dim=-1)
        correct += (preds == yb).sum().item()
        total += xb.size(0)
        for t, p in zip(yb.tolist(), preds.tolist()):
            per_class[t][1] += 1
            if t == p:
                per_class[t][0] += 1
    return correct / total, dict(per_class)


print('Helpers defined.')

Helpers defined.


## Build model, load pretrained, setup optimizer

In [8]:
# ============================================
# Cell 8 — Build model, load pretrained, training setup
# ============================================
NUM_CLASSES = 40
EPOCHS = 25
LR_INIT = 1e-4                    # low for fine-tuning
LR_MIN  = 1e-6
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
EMA_DECAY = 0.999

model = PointNetPlusPlusMSG(num_classes=NUM_CLASSES, dropout=0.5).to(device)
print(f'Model params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M\n')

print('Loading pretrained domain_aug backbone...')
load_pretrained(ckpt_path, model, device)

optimizer = AdamW(model.parameters(), lr=LR_INIT, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
loss_fn   = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
scaler    = torch.amp.GradScaler('cuda')
ema       = ModelEMA(model, decay=EMA_DECAY)

per_epoch_batches = len(train_loader)
print(f'\nTraining setup OK.')
print(f'  Epochs:           {EPOCHS}')
print(f'  Batches/epoch:    {per_epoch_batches}')
print(f'  LR:               {LR_INIT} -> {LR_MIN} (cosine)')
print(f'  EMA:              {EMA_DECAY}, AMP: on')
print(f'  Estimated time:   ~{EPOCHS * per_epoch_batches * 2 / 3600:.1f} hours')

Model params: 1.75M

Loading pretrained domain_aug backbone...
  Loading [model] from pointnetpp_msg_domain_aug_best.pt
  Loaded 163/163 weight tensors

Training setup OK.
  Epochs:           25
  Batches/epoch:    674
  LR:               0.0001 -> 1e-06 (cosine)
  EMA:              0.999, AMP: on
  Estimated time:   ~9.4 hours


## Training loop with checkpointing

In [9]:
# ============================================
# Cell 9 — Training loop (25 epochs, ~10 hours)
# ============================================
OUT_DIR = Path('/kaggle/working')
OUT_DIR.mkdir(exist_ok=True)

best_test_acc = 0.0
history = []

for ep in range(EPOCHS):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, scaler, loss_fn, ema, device)

    eval_model = PointNetPlusPlusMSG(num_classes=NUM_CLASSES).to(device)
    ema.apply(eval_model)
    te_acc, te_pc = evaluate(eval_model, test_loader, device)

    current_lr = scheduler.get_last_lr()[0]
    scheduler.step()
    elapsed = time.time() - t0

    print(f'Epoch {ep+1:3d}/{EPOCHS}  lr={current_lr:.6f}  '
          f'train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  '
          f'SO_test_acc(EMA)={te_acc:.4f}  ({elapsed:.0f}s)',
          flush=True)

    history.append({
        'epoch':     ep + 1,
        'lr':        current_lr,
        'train_loss':tr_loss,
        'train_acc': tr_acc,
        'test_acc':  te_acc,
    })

    if te_acc > best_test_acc:
        best_test_acc = te_acc
        torch.save({
            'epoch':       ep + 1,
            'model':       eval_model.state_dict(),
            'val_acc':     te_acc,
            'is_ema':      True,
            'num_classes': NUM_CLASSES,
        }, OUT_DIR / 'pointnetpp_msg_multi_domain_best.pt')
        print(f'   --> New best, saved (test_acc={te_acc:.4f})', flush=True)

    if (ep + 1) % 5 == 0 or ep == EPOCHS - 1:
        torch.save({
            'epoch': ep + 1, 'model': model.state_dict(),
            'val_acc': te_acc, 'is_ema': False,
        }, OUT_DIR / 'pointnetpp_msg_multi_domain_last.pt')
        with open(OUT_DIR / 'training_history.json', 'w') as f:
            json.dump(history, f, indent=2)

print(f'\nDone. Best test accuracy (SO PB_T50_RS): {best_test_acc:.4f}', flush=True)

Epoch   1/25  lr=0.000100  train_loss=2.7512  train_acc=0.3829  SO_test_acc(EMA)=0.4124  (433s)
   --> New best, saved (test_acc=0.4124)
Epoch   2/25  lr=0.000100  train_loss=2.1984  train_acc=0.4880  SO_test_acc(EMA)=0.4979  (437s)
   --> New best, saved (test_acc=0.4979)
Epoch   3/25  lr=0.000098  train_loss=2.0294  train_acc=0.5335  SO_test_acc(EMA)=0.5406  (437s)
   --> New best, saved (test_acc=0.5406)
Epoch   4/25  lr=0.000097  train_loss=1.9253  train_acc=0.5683  SO_test_acc(EMA)=0.5648  (437s)
   --> New best, saved (test_acc=0.5648)
Epoch   5/25  lr=0.000094  train_loss=1.8519  train_acc=0.5874  SO_test_acc(EMA)=0.5787  (437s)
   --> New best, saved (test_acc=0.5787)
Epoch   6/25  lr=0.000091  train_loss=1.8013  train_acc=0.6080  SO_test_acc(EMA)=0.5830  (437s)
   --> New best, saved (test_acc=0.5830)
Epoch   7/25  lr=0.000087  train_loss=1.7574  train_acc=0.6217  SO_test_acc(EMA)=0.5931  (437s)
   --> New best, saved (test_acc=0.5931)
Epoch   8/25  lr=0.000082  train_loss=1.7

## Final per-class evaluation on ScanObjectNN test

In [10]:
# ============================================
# Cell 10 — Final per-class evaluation
# ============================================
best_ckpt = torch.load(OUT_DIR / 'pointnetpp_msg_multi_domain_best.pt',
                       map_location=device, weights_only=False)
eval_model = PointNetPlusPlusMSG(num_classes=NUM_CLASSES).to(device)
eval_model.load_state_dict(best_ckpt['model'])
eval_model.eval()

test_acc, per_class = evaluate(eval_model, test_loader, device)
print(f'Best EMA test accuracy (SO PB_T50_RS mappable): {test_acc:.4f}\n')

print(f'{"MN40 class":15s}  {"correct":>8s} / {"total":>5s}   {"acc":>6s}')
print('-' * 50)
for i in range(NUM_CLASSES):
    if i in per_class:
        c, t = per_class[i]
        acc = c / t if t else 0
        print(f'{MODELNET40_CLASSES[i]:15s}  {c:>8d} / {t:>5d}   {acc:>6.1%}')

mean_class_acc = np.mean([c/t for c, t in per_class.values() if t > 0])
print(f'\nOverall accuracy:        {test_acc:.4f}')
print(f'Mean per-class accuracy: {mean_class_acc:.4f}')

Best EMA test accuracy (SO PB_T50_RS mappable): 0.6435

MN40 class        correct / total      acc
--------------------------------------------------
bed                    72 /   110    65.5%
bookshelf             152 /   241    63.1%
chair                 322 /   390    82.6%
desk                   75 /   150    50.0%
door                  151 /   210    71.9%
monitor               102 /   204    50.0%
sink                   53 /   120    44.2%
sofa                  159 /   210    75.7%
table                 136 /   270    50.4%
toilet                 59 /    85    69.4%
wardrobe              239 /   372    64.2%

Overall accuracy:        0.6435
Mean per-class accuracy: 0.6245
